## Overview
This notebook implements an automated pipeline to solve the **ML-CUP25 Regression Task** using an **Ensemble of Small Models**.

## Workflow
1.  **Data Loading**: 
    - Loads `ML-CUP25-TR.csv` and splits it into **Train**, **Validation** (20%), and **Blind Test**.

2.  **Architecture Search (Optuna)**: 
    - We use **Optuna** to find the best hyperparameters for a *single* small model.
    - **Constraints**: 
        - Depth: 1 to 2 hidden layers.
        - Width: 8 to 64 neurons per layer.
    - **Goal**: Minimize MSE on the Validation Set.

3.  **Ensemble Training**: 
    - We take the *best* architecture found by Optuna.
    - We instantiate **5 independent copies** of this model (the "Ensemble").
    - Each copy is trained from scratch with different random initializations.

4.  **Final Evaluation**:
    - The ensemble's prediction is the **average** of all 5 members.
    - We report the final MSE on the Validation and Test sets.

## How to Use
1.  **Prerequisites**: Ensure `data_loader.py` and the `models/` folder are in the same directory.
2.  **Configuration**: Adjust the constants in the first code cell (e.g., `N_TRIALS`, `ENSEMBLE_SIZE`) to control runtime.
3.  **Run All**: Execute all cells in order. The Optuna search will run first, followed automatically by the ensemble training.

In [ ]:
#--- 0. IMPORTS ---
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
import matplotlib.pyplot as plt
import sys
from sklearn.decomposition import PCA

# Ensure we can import from local modules
sys.path.append(os.getcwd())

# Import your custom modules
from utils.data_loader import get_ml_cup_data
from models import StandardFeedForwardNet, EnsembleModel
from models.ensemble import ModelWithHead, ReadoutAdapter
import training_utils

# --- CONFIGURATION ---
N_TRIALS = 1000
N_EPOCHS_SEARCH = 500
BATCH_SIZE = 64
N_EPOCHS_FINAL = 1000 # Train the final ensemble longer
ENSEMBLE_SIZE = 10   # How many models to stack together
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {DEVICE}")

OSError: [WinError 1114] Routine di inizializzazione della libreria di collegamento dinamico (DLL) non riuscita. Error loading "c:\Users\Michael\Desktop\Rage-Against-ML\.venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# --- 1. DATA LOADING ---
# train_loader, val_loader, test_loader, input_size, output_size

train_loader, val_loader, test_loader, INPUT_SIZE, OUTPUT_SIZE, target_scaler = get_ml_cup_data(
    BATCH_SIZE, 
    validation_ratio=0.15,
    test_ratio=0.10,
    scale_target=False,
    num_workers=0
)


# ==========================================
# PCA IMPLEMENTATION
# ==========================================
N_COMPONENTS = 2 # Set this to your desired number (e.g., 10). Set to 0 to disable.

if N_COMPONENTS > 0:
    print(f"Applying PCA with {N_COMPONENTS} components...")
    
    # 1. Helper to get numpy arrays from current loaders
    # Note: We assume data is already scaled by get_ml_cup_data (default behavior)
    X_train = train_loader.dataset.X.numpy()
    X_val   = val_loader.dataset.X.numpy()
    X_test  = test_loader.dataset.X.numpy()
    
    # 2. Fit PCA on Training Data ONLY
    pca = PCA(n_components=N_COMPONENTS)
    X_train_pca = pca.fit_transform(X_train)
    
    # 3. Transform Validation and Test
    X_val_pca  = pca.transform(X_val)
    X_test_pca = pca.transform(X_test)
    
    # 4. Update the Datasets with new features
    train_loader.dataset.X = torch.tensor(X_train_pca, dtype=torch.float32)
    val_loader.dataset.X   = torch.tensor(X_val_pca, dtype=torch.float32)
    test_loader.dataset.X  = torch.tensor(X_test_pca, dtype=torch.float32)
    
    # 5. CRITICAL: Update Input Size for the Model
    INPUT_SIZE = N_COMPONENTS
    print(f"PCA Applied. New Input Size: {INPUT_SIZE}")
    print(f"Explained Variance: {np.sum(pca.explained_variance_ratio_):.4f}")

else:
    print("PCA Disabled. Using original features.")

print(f"Data Loaded. Input Size: {INPUT_SIZE}, Output Size: {OUTPUT_SIZE}")
print(f"Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}")

In [ ]:
# --- 1.1 TARGET DISTRIBUTION ANALYSIS (Scaled vs Unscaled) ---
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# 1. Recover Raw Targets from Train Loader
# We extract the Y tensor and move to numpy
y_train_scaled = train_loader.dataset.y.numpy()

# If targets were scaled in loader, we inverse transform them to get raw
if target_scaler:
    y_train_raw = target_scaler.inverse_transform(y_train_scaled)
else:
    y_train_raw = y_train_scaled

# 2. Define Scalers to Compare
scalers = {
    "Unscaled (Raw)": None,
    "StandardScaler (Z-score)": StandardScaler(),
    "MinMaxScaler (0-1)": MinMaxScaler(),
    "RobustScaler (Quantile)": RobustScaler()
}

# 3. Plot Distributions for each Target Dimension (Y1 to Y4)
num_targets = y_train_raw.shape[1]
fig, axes = plt.subplots(num_targets, len(scalers), figsize=(20, 4 * num_targets))

print("Generating Target Distribution Plots and Statistics...")

# Set pandas display options for better readability of the describe() output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.3f}'.format)

for dim_i in range(num_targets):
    target_name = f"Target_Y{dim_i+1}"
    
    # Dictionary to collect data for this target across all scalers for describe()
    comparison_data = {}

    for idx, (scaler_name, scaler_obj) in enumerate(scalers.items()):
        ax = axes[dim_i, idx]
        
        # Transform data
        if scaler_obj is None:
            data_to_plot = y_train_raw[:, dim_i]
        else:
            # Fit on full raw y and transform (we fit on all targets, then extract the specific dim)
            transformed_y = scaler_obj.fit_transform(y_train_raw)
            data_to_plot = transformed_y[:, dim_i]
            
        # Add to comparison dictionary
        comparison_data[scaler_name] = data_to_plot
        
        # Plot Histogram & KDE
        sns.histplot(data_to_plot, kde=True, ax=ax, color='skyblue', edgecolor='black')
        
        # Formatting
        ax.set_title(f"{target_name} | {scaler_name}")
        ax.set_xlabel("Value")
        ax.set_ylabel("Frequency")
        
        # Add stats annotation (Mean/Std) to the plot
        mu = np.mean(data_to_plot)
        std = np.std(data_to_plot)
        ax.annotate(f"Mean: {mu:.2f}\nStd: {std:.2f}", 
                    xy=(0.05, 0.95), xycoords='axes fraction', 
                    bbox=dict(boxstyle="round", fc="white", alpha=0.8),
                    verticalalignment='top')
    
    # --- PRINT DESCRIBE() TABLE FOR CURRENT TARGET ---
    print(f"\n{'='*20} STATISTICS FOR {target_name} {'='*20}")
    df_compare = pd.DataFrame(comparison_data)
    print(df_compare.describe())
    print("-" * 80)

plt.tight_layout()
plt.show()

In [ ]:
# --- 2. RUN OPTUNA SEARCH & ANALYZE TOP 5 ---
import sys
import optuna
import pandas as pd
from IPython.display import display

# 1. Run the external script
# The script now self-cleans the lock upon exit
!{sys.executable} optuna_search_mlcup_nn.py --n_trials {N_TRIALS} --epochs {N_EPOCHS_SEARCH}

# 2. Access Results Safely
# We use a separate storage object so we can dispose of it explicitly
db_url = "sqlite:///optuna_mlcup_nn.db"
storage = optuna.storages.RDBStorage(url=db_url)

try:
    study = optuna.load_study(
        study_name="mlcup_search",
        storage=storage
    )

    # 3. Filter and Sort for the Top 5 Models
    df = study.trials_dataframe()
    df = df[df.state == "COMPLETE"]
    top_5 = df.sort_values('value', ascending=True).head(5)
    cols_to_show = ['number', 'value'] + [c for c in df.columns if c.startswith('params_')]

    print(f"\nSearch Complete. Best Params: {study.best_params}")
    print("\n--- TOP 5 BEST ARCHITECTURES FOUND ---")
    display(top_5[cols_to_show])
    
    # Store best params in a variable before closing study
    best_params = study.best_params

finally:
    # --- Release the file lock ---
    print("Disposing storage engine to release file lock...")
    storage.engine.dispose()
    del storage

In [ ]:
# --- 3. TRAINING ENSEMBLE ---
import copy
from tqdm import tqdm
from models.standard import StandardFeedForwardNet
from models.ensemble import ModelWithHead, ReadoutAdapter
from models.ensemble import EnsembleModel

print(f"\nTraining Ensemble of {ENSEMBLE_SIZE} models with best configuration...")

# 1. Detect Architecture & Extract Params
params = study.best_params

# We are forcing Standard MLP now
model_type = "standard" 
lr = params['lr']
weight_decay = params.get('weight_decay', 0.0)

print(f"Detected Architecture: {model_type.upper()}")

members = []
optimizers = []

for i in range(ENSEMBLE_SIZE):
    # 2. Instantiate Base Model
    # Reconstruct list of hidden sizes based on the new search logic
    n_layers = 2 # params['n_layers']
    hidden_size = params['hidden_size']
    hidden_sizes = [hidden_size] * n_layers
    
    base = StandardFeedForwardNet(
        input_size=INPUT_SIZE,
        hidden_sizes=hidden_sizes,
        output_size=OUTPUT_SIZE,
        activation=params['activation'],
        dropout=params.get('dropout', 0.0)
    )

    # 3. Wrap with Head (for Ensemble consistency)
    model = ModelWithHead(base, ReadoutAdapter(OUTPUT_SIZE, OUTPUT_SIZE, 'regression')).to(DEVICE)
    members.append(model)

    # 4. Setup Optimizer
    opt = optim.SGD(
        model.parameters(), 
        lr=lr, 
        weight_decay=weight_decay,
        momentum=0.9,    # Consistent with search
        nesterov=True    # Consistent with search
    )
    optimizers.append(opt)

# 5. Create the Ensemble Container
ensemble = EnsembleModel(members, weights=[1.0]*ENSEMBLE_SIZE).to(DEVICE)
criterion = nn.MSELoss()

train_losses = []
val_mses = []

# --- CHECKPOINTING INIT ---
best_ensemble_mse = float('inf')
best_epoch = -1
best_member_weights = [None] * ENSEMBLE_SIZE 

# Progress Bar
pbar = tqdm(range(N_EPOCHS_FINAL), desc="Training Ensemble", unit="epoch")

for epoch in pbar:
    ensemble.train() 
    total_loss = 0
    
    # --- TRAINING (On Scaled Data) ---
    for data, target in train_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        
        batch_loss = 0
        for model, opt in zip(members, optimizers):
            opt.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            opt.step()
            batch_loss += loss.item()
        
        total_loss += batch_loss / ENSEMBLE_SIZE
        
    current_train_loss = total_loss / len(train_loader)
    train_losses.append(current_train_loss)
    
    # --- VALIDATION (On Unscaled/Real Data) ---
    current_val_mse = training_utils.evaluate(
        ensemble, 
        val_loader, 
        criterion, 
        DEVICE, 
        target_scaler=target_scaler
    )
    val_mses.append(current_val_mse)
    
    # --- CHECKPOINTING LOGIC ---
    if current_val_mse < best_ensemble_mse:
        best_ensemble_mse = current_val_mse
        best_epoch = epoch
        
        for i, model in enumerate(members):
            best_member_weights[i] = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
            
        pbar.set_postfix({
            'Train': f"{current_train_loss:.4f}",
            'Val': f"{current_val_mse:.4f}",
            'Best': f"{best_ensemble_mse:.4f} (*)"
        })
    else:
        pbar.set_postfix({
            'Train': f"{current_train_loss:.4f}",
            'Val': f"{current_val_mse:.4f}",
            'Best': f"{best_ensemble_mse:.4f}"
        })

# --- RESTORE BEST WEIGHTS ---
print(f"\nTraining finished. Restoring best model from Epoch {best_epoch} (MSE: {best_ensemble_mse:.4f})...")

for i, model in enumerate(members):
    model.load_state_dict(best_member_weights[i])
    model.to(DEVICE)

print("Best ensemble weights restored and ready for testing.")

In [ ]:
import matplotlib.pyplot as plt

# --- 4. PLOT CONVERGENCE ---
print("\nPlotting convergence graph...")

plt.figure(figsize=(10, 6))
ax1 = plt.gca()

# Plot Training Loss (Scaled) on Left Y-Axis
color = 'tab:blue'
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Train Loss (Scaled)', color=color, fontweight='bold')
line1 = ax1.plot(train_losses, color=color, label='Train Loss (Scaled)', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

# Create Second Y-Axis for Validation MSE (Real)
ax2 = ax1.twinx()  
color = 'tab:orange'
ax2.set_ylabel('Val MSE (Real)', color=color, fontweight='bold')
line2 = ax2.plot(val_mses, color=color, label='Val MSE (Real)', linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

# Combine legends
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2)

plt.title('Ensemble Training Convergence', pad=30)
plt.tight_layout()

plt.show()

In [ ]:
# --- 5. RESULTS, PLOT & MEE CALCULATION ---
import numpy as np
import matplotlib.pyplot as plt
import torch

# 1. Plot Training History
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Avg Member Train Loss')
plt.plot(val_mses, label='Ensemble Val MSE (Real)', linewidth=2)
plt.title(f"Ensemble Training: {ENSEMBLE_SIZE} Models")
plt.xlabel("Epoch")
plt.ylabel("Error")
plt.legend()
plt.grid(True)
plt.show()

# 2. Final Evaluation on INTERNAL TEST SET (MEE Calculation)
print("Evaluating Ensemble on Internal Test Set...")

ensemble.eval()
total_mee = 0.0
total_samples = 0

# We'll keep track of the last batch to print ranges at the end
last_target_real = None 

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(DEVICE), target.to(DEVICE)
        
        # Get ensemble predictions (already averaged inside the model)
        out = ensemble(data)
        
        # --- INVERSE TRANSFORM TO REAL UNITS ---
        # Move to CPU and numpy
        out_np = out.cpu().numpy()
        target_np = target.cpu().numpy()
        
        if target_scaler is not None:
            # If scaler exists, reverse the scaling
            out_real = target_scaler.inverse_transform(out_np)
            target_real = target_scaler.inverse_transform(target_np)
        else:
            # If no scaler (None), data is already real
            out_real = out_np
            target_real = target_np
            
        last_target_real = target_real # Save for printing later
        
        # --- CALCULATE MEE (Mean Euclidean Error) ---
        # 1. Difference vector
        diff = out_real - target_real
        
        # 2. Euclidean distance for each sample: sqrt(sum(diff^2))
        # axis=1 sums across the output dimensions (e.g., x and y coordinates)
        euclidean_dists = np.sqrt(np.sum(diff**2, axis=1))
        
        # 3. Accumulate sum of distances
        total_mee += np.sum(euclidean_dists)
        total_samples += data.size(0)

# Final Average
final_mee = total_mee / total_samples

print("\n" + "="*40)
print(f"FINAL TEST RESULT")
print(f"Mean Euclidean Error (MEE): {final_mee:.4f}")
print("="*40)

# Check the range of the REAL targets (using the last batch processed)
if last_target_real is not None:
    print("Target Range (Y1):", f"{last_target_real[:, 0].min():.2f}", "to", f"{last_target_real[:, 0].max():.2f}")
    print("Target Range (Y2):", f"{last_target_real[:, 1].min():.2f}", "to", f"{last_target_real[:, 1].max():.2f}")
    print("Target Range (Y3):", f"{last_target_real[:, 2].min():.2f}", "to", f"{last_target_real[:, 2].max():.2f}")
    print("Target Range (Y4):", f"{last_target_real[:, 3].min():.2f}", "to", f"{last_target_real[:, 3].max():.2f}")
print("="*40)